Import cell performs the following key steps:

1. **Suppresses unnecessary output:** Disables logging messages at the WARNING level or above to prevent cluttering the console.
   - Key considerations:
     - If you need logging for debugging or more detailed information, adjust the logging level accordingly (e.g., `logging.INFO`).
     - Consider alternative logging configurations (e.g., file logging) for production environments.

2. **Imports required packages:** Ensures all necessary libraries are available for data manipulation, model building, and training.
   - The specific imports may vary depending on your project's requirements.

3. **Sets pandas display options:** Configures pandas to display all columns in DataFrames for comprehensive data exploration.
   - You might want to adjust this based on DataFrame sizes and your preferences.

4. **Sets maximum CPU count (optional):** Defines the number of CPU cores to be used for parallel processing.
   - This could be particularly relevant for computationally intensive tasks like model training.
   - Adjust the value based on your hardware and desired performance.
   - Consider using tools like `multiprocessing` or `concurrent.futures` for more granular control over parallelization.


In [ ]:
import logging

# Disable logging to suppress unnecessary output
logging.disable(logging.WARNING)

# Import required packages
from TL_Packages import *  # Assuming TL_Packages contains your custom functions/classes
from datetime import datetime
from sklearn.model_selection import KFold, train_test_split
import tensorflow as tf
import keras
from keras.models import load_model
import numpy as np
import os
import pandas as pd

# Set pandas option to display all columns
pd.set_option("display.max_columns", None)

# Set the maximum CPU count for parallel processing (optional)
os.environ["LOKY_MAX_CPU_COUNT"] = "4"

Cell performs the following tasks related to setting up the environment for machine learning:

1. **Sets Execution Directory:**
   - Defines a variable `execution_path` that points to the directory where the script will be executed.
   - This might be useful for relative path calculations or locating resources within the project structure.
   - Adjust the path based on your project's organization.

2. **Verifies Keras and Pandas Versions:**
   - Prints the versions of Keras and Pandas currently installed.
   - **Important:** This script checks for specific versions of Keras (2.12.0) and Pandas (< 2.0.0) due to potential backward compatibility issues.
   - The provided Stack Overflow link https://stackoverflow.com/questions/40898914/python-keras-new-installation explains this issue in more detail.
   - If you encounter compatibility problems, consider installing the recommended versions or updating your code to address the specific issues.

3. **Prints Version Information:**
   - Prints the versions of NumPy (used for numerical computations) and TensorFlow (the deep learning framework).
   - This information is helpful for debugging and understanding the environment in which your code is running.

4. **Checks for GPU Availability:**
   - Uses `tf.config.list_physical_devices('GPU')` to check if a physical GPU is available in your system.
   - Then, `tf.test.is_built_with_cuda()` verifies if TensorFlow is built with CUDA support for GPU acceleration.
   - Based on these checks, a variable `device` is set to either "cuda" (if GPU is available) or "cpu".
   - This allows the script to choose the appropriate device for computations, potentially improving performance with a GPU.

In [ ]:
# SET DIRECTORY
execution_path = "../../../"

# CHECK FOR CORRECT KERAS AND PANDAS VERSIONS
print("keras.__version__ = ", keras.__version__)  # 2.12.0

# Error will occur if pandas greater than specified due to loss of backward compatibility
# https://stackoverflow.com/questions/75953279/modulenotfounderror-no-module-named-pandas-core-indexes-numeric-using-metaflo
# pip install "pandas<2.0.0"
print("pd.__version__ = ", pd.__version__)  # 1.4.0 should be <2
print("np.__version__ = ", np.__version__)
print("tf.__version__ = ", tf.__version__)  # 2.10.0
# [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
print("tf.config.list_physical_devices('GPU') = ", tf.config.list_physical_devices())
print("tf.test.is_built_with_cuda() = ", tf.test.is_built_with_cuda())  # True
device = "cuda" if tf.test.is_built_with_cuda() else "cpu"
print("device = ", device)

Setting up global variables and file paths for a machine learning experiment that involves:

- Data loading and preprocessing
- Model training and evaluation
- Data augmentation techniques
- Population-based and patient-specific models

**Key components:**

1. **Data settings:**
   - Specifies the source data file and quantile.
   - Controls whether to remove features for age, BMI, and sex.
   - Defines labels for different patient groups.

2. **Model configuration:**
   - Determines model architecture (frozen layers).
   - Selects the data augmentation algorithm.
   - Controls model execution for population and individual models.
   - Sets cross-validation parameters for individual models.

3. **Experiment settings:**
   - Specifies the output folder for results.
   - Defines epoch sizes for population and patient models.
   - Specifies evaluation metrics to track model performance.

4. **Patient groups:**
   - Manages target groups for population and individual model training.

**Specific comments:**

- `source_file_path`: Path to the source data file.
- `Architecture_GROUP`: Type of model architecture to use.
- `augmentation_algo`: Data augmentation algorithm to apply.
- `bool_testing`: Flag for using a small sample of patients for testing.
- `bool_remove_target`: Controls whether to remove target data for patient-wise population models.
- `bool_drop_ABS`: Determines whether to remove age, BMI, and sex features.
- `bool_execute_pop_model`, `bool_execute_ind_model`: Flags for executing population and individual models.
- `num_pop_folds`, `num_ind_folds`: Number of folds for cross-validation in population and individual models.
- `output_folder`: Output directory for results.
- `RESULT_DIR`: Path to the results directory.
- `minority_variable`: Specifies the minority class for handling class imbalance.
- `CLASS_VAR`: Name of the class variable in the data.
- `POPULATION_EPOCH_SIZE_LIST`, `PATIENT_EPOCH_SIZE_LIST`: Lists of epoch sizes to consider for population and patient models.
- `pop_target_grp`: Target groups for population model training.
- `ind_target_grp`: Target groups for individual model training.
- `EVALUATION_METRICS`: List of metrics used for model evaluation.

In [3]:
# SETTING GLOBALS AND FILEPATHS

# Data to use:
source_quantile = "0.2"
source_file_path = Path(
    f"{execution_path}Input_Data/Og_Data/labeled_normal_data/all_pids_q={source_quantile}.npy"
)
print("source_file_path", source_file_path)
# Possible Values: "TL_1_Frozen", "TL_2_Frozen", "TL_3_Frozen", "TL_All_Frozen", "TL_All_Unfrozen"
Architecture_GROUP = "TL_All_Unfrozen"

# CHANGE ACCORDING TO YOUR ALGO
# Possible Values: ADKNN , GammaSmoteBoost,  <<-- Other Teams append their Algorithms here -->>
augmentation_algo = "GammaSmoteBoost"
augmentation_algo = "ADKNN"
augmentation_algo = "SMOTEBoostCC"
augmentation_algo = "SDDSMOTEBoostCC"

# When True only small sample of patients will run for testing purpose else All Patients as per parameters
bool_testing = False

# WHEN FALSE, TARGET DATA IS NOT REMOVED FROM ALL PATIENTS DATA
# False for Generalized model, True for Patient-wise Pop Models
bool_remove_target = False

# When True will drop Age, BMI and Sex columns too else will keep them too
bool_drop_ABS = True

# When True will execute population_model else do not execute
# Note: population_model is generally executed only once and its output is reused multiple times
bool_execute_pop_model = False

# Number of Folds for running Pop Model
# Possible values : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
num_pop_folds = 1

# When True will execute individual_model else do not execute
bool_execute_ind_model = True

# Number of Folds for running Ind Model
# Possible values : [3, 4, 5]
num_ind_folds = 1

## Use True if you choose to not have cv for your individual models
bool_remove_ind_cv = True

# Change output folder during each run to not overwrite previous outputs\
# Should not include start or end '/'
output_folder = "SDDSmoteBoost_Unf_All_Patients"

if bool_drop_ABS == False:
    output_folder += "_ABS"

RESULT_DIR = Path(f"{execution_path}/Output_Results/{output_folder}/")
print("RESULT_DIR", RESULT_DIR)

minority_variable = 0
CLASS_VAR = "class"


POPULATION_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]
PATIENT_EPOCH_SIZE_LIST = [100, 300, 500, 700, 1000, 2000, 3000]

""" Removing SB-081 due to less data """
""" Removing SB-033 due to less data """
""" Removing SB-089 due to less data """

if bool_remove_target == True:
    pop_target_grp = [
        "SB-001",
        "SB-003",
        "SB-008",
        "SB-011",
        "SB-012",
        "SB-022",
        "SB-023",
        "SB-025",
        "SB-028",
        "SB-037",
        "SB-043",
        "SB-056",
        "SB-059",
        "SB-060",
        "SB-070",
        "SB-071",
        "SB-073",
        "SB-078",
        "SB-079",
        "SB-080",
        "SB-082",
        "SB-083",
        "SB-112",
    ]
else:
    pop_target_grp = ["source"]

# Used for individual model
ind_target_grp = [
    "SB-001",
    "SB-003",
    "SB-008",
    "SB-011",
    "SB-012",
    "SB-022",
    "SB-023",
    "SB-025",
    "SB-028",
    "SB-037",
    "SB-043",
    "SB-056",
    "SB-059",
    "SB-060",
    "SB-070",
    "SB-071",
    "SB-073",
    "SB-078",
    "SB-079",
    "SB-080",
    "SB-082",
    "SB-083",
    "SB-112",
]

EVALUATION_METRICS = [
    "Weighted Accuracy",
    "Sensitivity/Recall",
    "Specificity",
    "Precision_class0",
    "Precision_class1",
    "Precision_avg",
    "F1_class0",
    "F1_class1",
    "F1_avg",
    "auc_roc_score",
    "False_Discovery_Rate",
    "False_Negative_Rate",
    "False_Omission_Rate",
    "False_Positive_Rate",
    "Jaccard",
]

source_file_path ..\..\..\Input_Data\Og_Data\labeled_normal_data\all_pids_q=0.2.npy
RESULT_DIR ..\..\..\Output_Results\SDDSmoteBoost_Unf_All_Patient_without_CV


---

Prints the current time in a human-readable format.

This function retrieves the current time using the `datetime` module
and then formats it as "Runtime: {hour} o' clock and {minute} minutes".

For example, if the current time is 3:25 PM, the output would be:
    Runtime: 3 o' clock and 25 minutes

In [4]:
def runtime():
    """
    Prints the current time in a human-readable format.
    """

    # Get the current hour (0-23) and minute (0-59)
    current_hour = datetime.datetime.now().hour
    current_minute = datetime.datetime.now().minute

    # Print the formatted time message
    print(f"Runtime: {current_hour} o' clock and {current_minute} minutes")

Loads data from a file and converts it to a NumPy array (if possible).

In [5]:
# Function to load data from a file and convert it to a NumPy array (if applicable)
def load_data(source_file_path, allow_pickle=True):
    """Loads data from a file and converts it to a NumPy array (if possible).

    Args:
        source_file_path (str): Path to the file containing the data.
        allow_pickle (bool, optional): Whether to allow unpickling of data from the file
            (can be a security risk if the source is untrusted). Defaults to True.

    Returns:
        object: The loaded data, which may be a NumPy array or the original data type
            if conversion is not possible.

    Raises:
        IOError: If an error occurs while loading the data from the file.
    """

    try:
        # Load data from the file using np.load with allow_pickle for flexibility
        data_ndarr = np.load(source_file_path, allow_pickle=allow_pickle)
        print("data_ndarr type:", type(data_ndarr))  # Print for debugging/verification

        # Extract the data dictionary from the NumPy array (assuming it holds a single element)
        datadict = data_ndarr.item()
        print("datadict type:", type(datadict))  # Print for debugging/verification
        return datadict  # Return the extracted dictionary

    except IOError as e:
        raise IOError(f"Error loading data from file: {source_file_path}") from e


# Example usage (assuming source_file_path is set to a valid file)
datadict = load_data(source_file_path)

data_ndarr type : <class 'numpy.ndarray'>
datadict type : <class 'dict'>


----

This function processes a dictionary of patient data frames (`datadict`), dropping
unnecessary columns and converting the remaining data to NumPy arrays for efficient
numerical operations.

Args:
    datadict (dict): A dictionary where keys are patient IDs and values are pandas DataFrames
        containing patient data.
    bool_drop_ABS (bool, optional): A flag indicating whether to drop additional columns
        beyond the default set. Defaults to False.

Returns:
    None

Prints:
    - The number of patients in the dataset.
    - The shape of the processed data frame for each patient.
    - A sample of the head (first 5 rows) of the processed data frame for a single patient.

Raises:
    ValueError: If `bool_drop_ABS` is True and any columns other than `id`, `date`,
                 and `am_pef_org` are present in the data frames, as they might not be
                 convertible to NumPy float32.

In [ ]:
def process_patient_data(datadict, bool_drop_ABS=False):
    p_ids = datadict.keys()
    sample_size_dict = {}
    print("patients: n =", len(p_ids), end="\n\n")

    for p_id in p_ids:
        df = datadict[p_id]

        # Drop common unnecessary columns
        df.drop(["id", "date", "am_pef_org"], axis=1, inplace=True)

        # Optionally drop additional columns for further processing
        if bool_drop_ABS:
            valid_cols = set(df.columns) - {"id", "date", "am_pef_org"}
            if valid_cols:
                raise ValueError(
                    "Unexpected columns found with bool_drop_ABS=True: {}".format(
                        valid_cols
                    )
                )
            # No additional columns found, proceed with conversion

        # Convert data frame to NumPy array and cast to float32
        df = np.asarray(df).astype(np.float32)
        print(p_id, "shape:", df.shape)
        sample_size_dict[p_id] = df.shape[0]

    return sample_size_dict, p_ids, df


sample_size_dict, p_ids, df = process_patient_data(datadict, bool_drop_ABS=True)

patients: n = 26

SB-001 shape: (188, 28)
SB-003 shape: (180, 28)
SB-008 shape: (178, 28)
SB-011 shape: (210, 28)
SB-012 shape: (194, 28)
SB-022 shape: (204, 28)
SB-023 shape: (194, 28)
SB-025 shape: (117, 28)
SB-028 shape: (92, 28)
SB-033 shape: (53, 28)
SB-037 shape: (210, 28)
SB-043 shape: (191, 28)
SB-056 shape: (191, 28)
SB-059 shape: (203, 28)
SB-060 shape: (88, 28)
SB-070 shape: (202, 28)
SB-071 shape: (124, 28)
SB-073 shape: (161, 28)
SB-078 shape: (153, 28)
SB-079 shape: (142, 28)
SB-080 shape: (181, 28)
SB-081 shape: (34, 28)
SB-082 shape: (137, 28)
SB-083 shape: (206, 28)
SB-089 shape: (49, 28)
SB-112 shape: (180, 28)


,income,road_dist,cooking,y_am_pef,tempin,humidin,pm25in,co2in,tempdiffin,humidiffin,pm25diffin,pm10,pm25,o3,no2,co,so2,temp,windsd,humid,varp,dewpt,airp,seap,solrhr,solramnt,grdt,class
1,0.857,0.0,0.167,0.558,0.216,0.513,0.358,0.678,0.562,0.364,0.619,0.390,0.250,0.024,0.587,0.536,0.222,0.349,0.107,0.668,0.174,0.471,0.822,0.819,0.165,0.003,0.206,0.0
2,0.857,0.0,0.167,0.006,0.299,0.521,0.411,0.700,0.436,0.672,0.616,0.446,0.285,0.000,0.783,0.732,0.222,0.473,0.171,0.388,0.190,0.491,0.676,0.670,0.603,0.165,0.383,0.0
3,0.857,0.0,0.167,0.078,0.420,0.713,0.372,0.918,0.469,0.821,0.734,0.466,0.155,0.190,0.239,0.262,0.111,0.474,0.515,0.359,0.197,0.481,0.519,0.518,0.485,0.104,0.346,1.0
4,0.857,0.0,0.167,0.428,0.438,0.468,0.224,0.762,0.560,0.490,0.794,0.211,0.011,0.143,0.196,0.110,0.111,0.404,0.338,0.527,0.175,0.467,0.596,0.596,0.491,0.141,0.339,0.0
5,0.857,0.0,0.167,0.209,0.462,0.385,0.199,0.502,0.612,0.529,0.725,0.250,0.057,0.095,0.348,0.245,0.111,0.366,0.316,0.444,0.141,0.419,0.765,0.762,0.433,0.135,0.281,1.0


"\n(Note: `am_pef_org` is the UNSHIFTED, NON-NORMALIZED value for a person's morning exposure. It is relatively raw, and should almost exclusively be used for classification purposes)\n"

---
### Description:
This function splits a given dataset into training and testing subsets according to the specified ratio.

### Arguments:
- `data` (DataFrame): The dataset to be split.
- `test_size` (float): The proportion of the dataset to include in the test split. Should be between 0.0 and 1.0.
- `random_state` (int, optional): Controls the randomness of the sample. Defaults to None.

### Returns:
tuple: A tuple containing the training dataset and the testing dataset, in that order.

### Examples:
```python
train_data, test_data = split_data(data, test_size=0.2, random_state=42)


In [ ]:
def split_data(
    target, data, bool_remove_target, transfer=False
) -> (pd.DataFrame, pd.DataFrame):
    """
    Creates a combined data set from all the ids in the given dictionary,
    If transfer==false, returns just target df
    If transfer==True and bool_remove_target==True, returns all but the target df.
    If transfer==True and bool_remove_target==False, return all patients’ data

    :param target: Who the target is for prediction.
    :param data: Dict of DataFrames for prediction.
    :param bool_remove_target: Boolean flag indicating whether to remove the target DataFrame or not.
    :param transfer: Boolean flag indicating whether to transfer data.
    :return X: Predictive labels.
    :return Y: Dependent label.
    """
    # Create an empty DataFrame to hold the combined data
    dataset = pd.DataFrame()

    # Check if data transfer is requested
    if transfer == True:
        # If transfer is requested and removal of target DataFrame is also requested
        if bool_remove_target == True:
            # Concatenate all DataFrames except the target DataFrame
            dataset = pd.concat(
                [data[key].copy() for key in data.keys() if key != target]
            )
        else:
            # Concatenate all DataFrames including the target DataFrame
            dataset = pd.concat([data[key].copy() for key in data.keys()])

    else:
        # If transfer is not requested, simply copy the target DataFrame
        dataset = data[target].copy()

    # Extract features (X) by dropping the target variable
    X = dataset.drop([CLASS_VAR], axis=1)
    # Extract the target variable (Y)
    Y = dataset.loc[:, CLASS_VAR:CLASS_VAR]

    return X, Y

---
### Description:
This function is used to augment a given DataFrame using a specified technique to balance the dataset, particularly when dealing with imbalanced classes. It allows for plugging in different augmentation algorithms by providing the appropriate function call and ensuring compatibility with the input and output formats.

### Arguments:
- `augmentation_algo` (str): Name of the augmentation technique to be applied.
- `train_dataset` (DataFrame): DataFrame representing the dataset to be augmented, containing both feature variables (X) and labels (Y). Certain columns are dropped: ["id", "date", "am_pef_org", "BMI", "sex", "age"].
- `classIndex` (str): Label of the class with minority/majority split.
- `minorityLabel` (str): The value of `classIndex` representing the minority class.
- `printDebug` (bool, optional): Flag for enabling debug printing. Defaults to False.

### Returns:
DataFrame: Augmented dataset, similar to `train_dataset` but with synthetic rows included to achieve balance.

### Notes:
- Other augmentation algorithms can be plugged in by ensuring compatibility with the function signature and return type.
- Different augmentation techniques are implemented based on the value of `augmentation_algo`.
- Upon successful augmentation, the function prints out the value counts of the class labels to ensure balance.
- If the specified augmentation technique is not implemented, a `NotImplementedError` is raised.

### Examples:
```python
augmented_df = data_augmentation("ADKNN", train_df, "label", "minority", printDebug=True)

In [8]:
def data_augmentation(
    augmentation_algo,
    train_dataset,
    classIndex,
    minorityLabel,
    printDebug=False,
):
    """
    Augments the given dataframe via the specified technique.

    Args:
        augmentation_algo (str): Name of the augmentation technique to be used.
        train_dataset (DataFrame): DataFrame to be augmented, containing both feature variables (X) and labels (Y).
                                   It represents the complete dataset except the train split.
                                   Certain columns are dropped: ["id", "date", "am_pef_org", "BMI", "sex", "age"].
        classIndex (str): Label of the class with minority/majority split.
        minorityLabel (str): The value of classIndex representing the minority class.
        printDebug (bool, optional): Flag for enabling debug printing. Defaults to False.

    Returns:
        DataFrame: Augmented dataset, similar to train_dataset but with synthetic rows included to achieve balance.

    Note:
        - Other augmentation algorithms can be plugged in by ensuring compatibility with the function signature and return type.
        - For the augmentation technique, 'ADKNN', the `balance_method.ADKNN` function is used.
        - For the augmentation technique, 'AutoEncoder_1', the `balance_method.generate_autoencoder` function is used.
        - For the augmentation technique, 'GammaSmoteBoost', the `balance_method.GammaSMOTEBoostCC` function is used.
        - For the augmentation technique, 'SDDSMOTEBoostCC', the `balance_method.SDDSMOTEBoostCC` function is used.
        - For the augmentation technique, 'SMOTEBoostCC', the `balance_method.SMOTEBoostCC` function is used.

    Raises:
        NotImplementedError: If augmentation_algo is not recognized.

    Examples:
        >>> augmented_df = data_augmentation("ADKNN", train_df, "label", "minority", printDebug=True)
    """
    if augmentation_algo == "ADKNN":
        train_dataset = balance_method.ADKNN(
            train_dataset,
            classIndex=classIndex,
            minorityLabel=minorityLabel,
            printDebug=printDebug,
        )
    elif augmentation_algo == "AutoEncoder_1":
        train_dataset = balance_method.generate_autoencoder(
            train_dataset,
            class_var=classIndex,
            minority_var=minorityLabel,
            printDebug=printDebug,
        )
    elif augmentation_algo == "GammaSmoteBoost":
        train_dataset = balance_method.GammaSMOTEBoostCC(
            train_dataset,
            numIterations=5,
            printDebug=printDebug,
        )
    elif augmentation_algo == "SDDSMOTEBoostCC":
        train_dataset = balance_method.SDDSMOTEBoostCC(
            train_dataset,
            printDebug=printDebug,
        )
    elif augmentation_algo == "SMOTEBoostCC":
        train_dataset = balance_method.SMOTEBoostCC(
            train_dataset,
            numIterations=5,
            printDebug=printDebug,
        )
        print(f"augmented data size: {train_dataset.shape}")
    else:
        raise NotImplementedError(
            f"Augmentation technique '{augmentation_algo}' not implemented."
        )

    print("~~~~~~~ value_counts: ~~~~~~~")
    print("SHOULD BE BALANCED NOW:\n", train_dataset[classIndex].value_counts())
    return train_dataset

---
### Description:
Trains a population-based source model.

### Arguments:
- `pop_target_grp` (list): List of targets to train for, e.g. ["SB-070"]
- `POPULATION_EPOCH_SIZE_LIST` (list): List of Number of epochs for NN
- `num_pop_folds` (int): Number of k-fold cross validations
- `bool_remove_target` (bool): Whether or not to remove target pt from source model training
- `augmentation_algo` (str): Name of the augmentation technique to be applied.

### Returns:
None, but save models to directory. P.S. Needs filepath names resolved above.

### Notes:
- Initializes the sample size.
- Loops through each target group and performs the following steps:
  1. Measures runtime.
  2. Defines a directory for the current patient.
  3. Splits data based on the target group.
  4. Performs k-fold cross-validation.
  5. Iterates over different epoch sizes for neural network training.
  6. Trains neural network, labels, and saves the model.
  7. Appends evaluation results to a dataframe.
  8. Writes evaluation results to a CSV file.

### Examples:
```python
population_model(
    pop_target_grp=["SB-070"],
    POPULATION_EPOCH_SIZE_LIST=[10, 20, 30],
    num_pop_folds=5,
    bool_remove_target=True,
    augmentation_algo="ADKNN"
)

In [9]:
def population_model(
    pop_target_grp,
    POPULATION_EPOCH_SIZE_LIST,
    # PATIENT_EPOCH_SIZE,
    num_pop_folds,
    bool_remove_target,
    augmentation_algo,
):
    """
    Trains a population-based source model.

    :param pop_target_grp: List of targets to train for, e.g. ["SB-070"]
    :param POPULATION_EPOCH_SIZE_LIST: List of Number of epochs for NN
    # :param PATIENT_EPOCH_SIZE: Used to label model for use later
    :param num_pop_folds: Number of k-fold cross validations
    :param bool_remove_target: Whether or not to remove target pt from source model training
    :return: None, but save models to directory. P.S. Needs filepath names resolved above.
    """

    sample_size = ""  # Initialize sample size

    # Loop through each target group
    for i in range(0, len(pop_target_grp)):
        target = pop_target_grp[i]  # Get current target group
        runtime()  # Measure runtime

        # Define directory for current patient
        patient_dir = Path(
            f"{RESULT_DIR}/{Architecture_GROUP}/diagrams_n_population_models/{target}"
        )

        # Create directory for this patient if it doesn't exist
        if not (patient_dir.exists() and patient_dir.is_dir()):
            os.makedirs(patient_dir)

        # Split data based on target group
        X, Y = split_data(
            target=target,
            data=datadict,
            bool_remove_target=bool_remove_target,
            transfer=True,
        )

        # Perform k-fold cross validation
        if num_pop_folds > 1:
            kf = KFold(n_splits=num_pop_folds, shuffle=False).split(X.to_numpy())

        for fold in range(1, num_pop_folds + 1):
            # Make train/test sets
            if num_pop_folds == 1:
                X_train, X_test, Y_train, Y_test = train_test_split(
                    X.to_numpy(), Y.to_numpy(), test_size=0.2, shuffle=True
                )
            else:
                train_index, test_index = next(kf)
                X_train, X_test = X.to_numpy()[train_index], X.to_numpy()[test_index]
                Y_train, Y_test = Y.to_numpy()[train_index], Y.to_numpy()[test_index]

            # Dataframe manipulation for training set
            df_X_train = pd.DataFrame(X_train, columns=X.columns.values)
            df_Y_train = pd.DataFrame(Y_train, columns=Y.columns.values)
            joined_train_dataset = df_X_train.join(df_Y_train)

            # Augment data
            train_dataset = data_augmentation(
                augmentation_algo,
                joined_train_dataset,
                classIndex=CLASS_VAR,
                minorityLabel=minority_variable,
            )

            # Convert augmented data back to numpy arrays
            X_train = train_dataset.drop(CLASS_VAR, axis=1).to_numpy()
            Y_train = train_dataset.loc[:, CLASS_VAR:CLASS_VAR].to_numpy()

            # Initialize dataframe for storing evaluation results
            source_evaluation_results = pd.DataFrame(columns=EVALUATION_METRICS)

            # Iterate over different epoch sizes for neural network training
            for POPULATION_EPOCH_SIZE in POPULATION_EPOCH_SIZE_LIST:
                print(
                    "~~~~~~~~~~~",
                    POPULATION_EPOCH_SIZE,
                    "~~~~~~~~~~~~~~~~~~~",
                    target,
                    "~~~~~~~~~~~~~~~~~~~",
                )

                # Train neural network
                print(f"Population Fold {fold}: Training data size = {len(X_train)}")

                # Initialize and train neural network
                net = network_tl(
                    X_train, Y_train, n_hidden=32, learning_rate=0.001, device=device
                )
                net.set_Y_tind(Y_train)
                net.make_pop(Architecture_GROUP)
                net.train(n_iters=POPULATION_EPOCH_SIZE)

                # Update sample size if target removal is not required
                if bool_remove_target == False:
                    sample_size = f"{str(len(X_train))}"

                # Label and save model
                VERSION_TAG = f"{target}_size_{sample_size}_{Architecture_GROUP}_{POPULATION_EPOCH_SIZE}_POP_FOLD_{fold}"
                modelname = f"{VERSION_TAG}.keras"
                training_curve_file = patient_dir / f"{VERSION_TAG}_training_curve"

                model_path = patient_dir / modelname
                net.nn_pop.save(model_path)  # nn_pop
                net.save_training_curve(training_curve_file)
                source_results, source_c_matrix = net.pop_test(X_test, Y_test)

                # Append evaluation results to dataframe
                source_evaluation_results.loc[f"{VERSION_TAG}"] = list(source_results)

            # Define output path for evaluation metrics
            if bool_remove_target == True:
                output_path = os.path.join(
                    f"{RESULT_DIR}/{Architecture_GROUP}/diagrams_n_population_models",
                    f"Pop_{Architecture_GROUP}_evaluation_metrics_TargetRemoved.csv",
                )
            else:
                output_path = os.path.join(
                    f"{RESULT_DIR}/{Architecture_GROUP}/diagrams_n_population_models",
                    f"Pop_{Architecture_GROUP}_evaluation_metrics.csv",
                )

            # Write evaluation results to CSV file
            if os.path.isfile(output_path):
                source_evaluation_results.to_csv(output_path, mode="a", header=False)
            else:
                source_evaluation_results.to_csv(output_path, mode="a")

    print("evaluation results saved")

---
### Description:
This function serves as a wrapper for executing the population model. It checks if testing mode is enabled and if the population model should be executed, and then calls the `population_model` function with the specified parameters.

### Arguments:
- `bool_testing` (bool): Flag indicating whether the code is running in testing mode.
- `bool_execute_pop_model` (bool): Flag indicating whether to execute the population model.
- `pop_target_grp` (str): Target group for population modeling.
- `POPULATION_EPOCH_SIZE_LIST` (List[int]): List of epoch sizes for population modeling.
- `num_pop_folds` (int): Number of folds for cross-validation in population modeling.
- `bool_remove_target` (bool): Flag indicating whether to remove the target variable.
- `augmentation_algo` (str): Name of the augmentation technique to be applied.

### Returns:
None

In [13]:
def population_model_wrapper(
    bool_testing: bool,
    bool_execute_pop_model: bool,
    pop_target_grp: str,
    POPULATION_EPOCH_SIZE_LIST: List[int],
    num_pop_folds: int,
    bool_remove_target: bool,
    augmentation_algo: str,
) -> None:
    """
    Wrapper function for executing population model.

    This function checks if testing mode is enabled and if the population model
    should be executed. If both conditions are met, it calls the population_model
    function with the specified parameters.

    Args:
    - bool_testing (bool): Flag indicating whether the code is running in testing mode.
    - bool_execute_pop_model (bool): Flag indicating whether to execute the population model.
    - pop_target_grp (str): Target group for population modeling.
    - POPULATION_EPOCH_SIZE_LIST (List[int]): List of epoch sizes for population modeling.
    - num_pop_folds (int): Number of folds for cross-validation in population modeling.
    - bool_remove_target (bool): Flag indicating whether to remove the target variable.
    - augmentation_algo (str): Name of the augmentation technique to be applied.

    Returns:
    None

    """
    if not bool_testing:  # If not in testing mode
        if bool_execute_pop_model:  # If population model execution is requested
            population_model(
                pop_target_grp,
                POPULATION_EPOCH_SIZE_LIST,
                num_pop_folds,
                bool_remove_target,
                augmentation_algo,
            )